In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import re
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Step 1: Load and Combine Agricultural Data
files = [
    '2013-2014.csv', '2014-2015.csv', '2015-2016.csv', '2016-2017.csv', 
    '2017-2018.csv', '2018-2019.csv', '2019-2020.csv', '2020-2021.csv', 
    '2021-2022.csv', '2022-2023.csv', '2023-2024.csv'
]

# Column name mapping to standardize variations
column_mapping = {
    r'(?i)^Paddy\s*(Area|area)\s*\(?[hH]a\)?$': 'Paddy_Area_ha',
    r'(?i)^Paddy\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Paddy_Production_MT',
    r'(?i)^Paddy\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Paddy_Yield',
    r'(?i)^Maize\s*(Area|area)\s*\(?[hH]a\)?$': 'Maize_Area_ha',
    r'(?i)^Maize\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Maize_Production_MT',
    r'(?i)^Maize\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Maize_Yield',
    r'(?i)^Wheat\s*(Area|area)\s*\(?[hH]a\)?$': 'Wheat_Area_ha',
    r'(?i)^Wheat\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Wheat_Production_MT',
    r'(?i)^Wheat\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Wheat_Yield',
    r'(?i)^District$': 'District',
    r'(?i)^Year$': 'Year'
}

dfs = []
for file in files:
    df = pd.read_csv(file)
    # Standardize column names using regex mapping
    new_columns = {}
    for col in df.columns:
        for pattern, standard_name in column_mapping.items():
            if re.match(pattern, col):
                new_columns[col] = standard_name
                break
        else:
            new_columns[col] = col  # Keep unchanged if no match
    df.rename(columns=new_columns, inplace=True)
    
    # Add Year if missing
    if 'Year' not in df.columns:
        df['Year'] = file.split('.')[0]
    
    # Standardize district names
    df['District'] = df['District'].str.upper().replace({
        'KAVREPALANCHOK': 'KAVRE',
        'NAWALPARASI_EAST': 'NAWALPUR_EAST',
        'NAWALPARASI_WEST': 'NAWALPUR_WEST',
        'EASTERN_RUKUM': 'RUKUM_EAST',
        'WESTERN_RUKUM': 'RUKUM_WEST',
        'NAWALPARASI_(WEST_OF_BARDAGHAT_SUSTA)': 'NAWALPUR_WEST',
        'SANKHUWASAVA': 'SANKHUWASABHA'
    })
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

# Calculate yields for files missing them
for crop in ['Paddy', 'Maize', 'Wheat']:
    if f'{crop}_Yield' not in data.columns or data[f'{crop}_Yield'].isna().any():
        if f'{crop}_Production_MT' in data.columns and f'{crop}_Area_ha' in data.columns:
            data[f'{crop}_Yield'] = data[f'{crop}_Production_MT'] / data[f'{crop}_Area_ha'].replace(0, np.nan)
        else:
            print(f"Warning: Missing {crop}_Production_MT or {crop}_Area_ha in some files")

# Handle missing values and zeros
data.replace(0, np.nan, inplace=True)
data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']] = data.groupby('District')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].transform(lambda x: x.fillna(x.median()))
data.dropna(subset=['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield'], inplace=True)

# Remove outliers
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column].isna()) | ((df[column] >= lower_bound) & (df[column] <= upper_bound))]

for col in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    data = remove_outliers(data, col)

# Step 2: Merge Climate Data
climate_data = pd.read_csv('Nepal_77districts_annual_climate.csv')
climate_data['District'] = climate_data['District'].str.upper().replace({'SANKHUWASAVA': 'SANKHUWASABHA'})
climate_data['Year'] = climate_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')
data = pd.merge(data, climate_data, on=['Year', 'District'], how='inner')

# Step 3: Distribute Fertilizer Data
fertilizer_data = pd.read_csv('fertilizer_annual_2013_2024.csv')
fertilizer_data['Year'] = fertilizer_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')
data['Total_Area'] = data[['Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha']].sum(axis=1)
data = data.merge(data.groupby('Year')['Total_Area'].sum().reset_index(name='National_Total_Area'), on='Year')
data['Area_Share'] = data['Total_Area'] / data['National_Total_Area']
data = pd.merge(data, fertilizer_data, on='Year', how='left')
for fert in ['Urea', 'DAP', 'Potash']:
    data[f'District_{fert}_MT'] = data[fert] * data['Area_Share']

# Step 4: EDA
# Yield trends
national_yields = data.groupby('Year')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].mean().reset_index()
plt.figure(figsize=(10, 6))
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.plot(national_yields['Year'], national_yields[crop], label=crop)
plt.xlabel('Year')
plt.ylabel('Yield (t/ha)')
plt.title('National Crop Yield Trends (2013-2024)')
plt.legend()
plt.xticks(rotation=45)
plt.savefig('yield_trends.png')
plt.close()

# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield', 'Annual_Mean_Temp', 
                  'Annual_Total_Precip', 'Monsoon_Total_Precip', 'District_Urea_MT', 
                  'District_DAP_MT', 'District_Potash_MT']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Between Features and Crop Yields')
plt.savefig('correlation_heatmap.png')
plt.close()

# Yield distributions
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.figure(figsize=(8, 6))
    sns.histplot(data[crop], bins=30, kde=True)
    plt.xlabel(f'{crop} (t/ha)')
    plt.title(f'Distribution of {crop}')
    plt.savefig(f'{crop}_distribution.png')
    plt.close()

# Step 5: Feature Selection
features = ['Annual_Mean_Temp', 'Annual_Total_Precip', 'Monsoon_Total_Precip', 
            'District_Urea_MT', 'District_DAP_MT', 'District_Potash_MT', 
            'Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha', 'Latitude', 'Longitude']
scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])

# Step 6: Model Development
target_crops = ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']
results = {}

for crop in target_crops:
    X = data[features]
    y = data[crop]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }
    grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    results[crop] = {
        'Best_Model': grid_search.best_estimator_, 
        'X_test': X_test, 
        'y_test': y_test,
        'Best_Params': grid_search.best_params_
    }

# Step 7: Model Evaluation
rmse_values = []
r2_values = []
for crop in target_crops:
    model = results[crop]['Best_Model']
    y_pred = model.predict(results[crop]['X_test'])
    rmse = np.sqrt(mean_squared_error(results[crop]['y_test'], y_pred))
    r2 = r2_score(results[crop]['y_test'], y_pred)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    
    rmse_values.append(rmse)
    r2_values.append(r2)
    
    print(f"{crop}:")
    print(f"  RMSE: {rmse:.3f} t/ha")
    print(f"  R2: {r2:.3f}")
    print(f"  Cross-Validation R2: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
    print(f"  Best Parameters: {results[crop]['Best_Params']}")
    
    # Feature importance
    importance = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_})
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Importance', y='Feature', data=importance.sort_values(by='Importance', ascending=False))
    plt.title(f'Feature Importance for {crop}')
    plt.savefig(f'feature_importance_{crop}.png')
    plt.close()

# Step 8: Scenario Analysis
paddy_model = results['Paddy_Yield']['Best_Model']
X_scenario = results['Paddy_Yield']['X_test'].copy()
scenarios = {
    'Baseline': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0},
    'Rainfall_+10%': {'Annual_Total_Precip': 0.1, 'District_Urea_MT': 0},
    'Rainfall_-10%': {'Annual_Total_Precip': -0.1, 'District_Urea_MT': 0},
    'Urea_+10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0.1},
    'Urea_-10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': -0.1}
}
scenario_results = {}

for scenario, changes in scenarios.items():
    X_temp = X_scenario.copy()
    for feature, change in changes.items():
        X_temp[feature] = X_scenario[feature] * (1 + change)
    scenario_results[scenario] = paddy_model.predict(X_temp).mean()

plt.figure(figsize=(8, 6))
plt.bar(scenario_results.keys(), scenario_results.values())
plt.ylabel('Predicted Paddy Yield (t/ha)')
plt.title('Scenario Analysis: Rainfall and Urea Impact')
plt.xticks(rotation=45)
plt.savefig('scenario_analysis.png')
plt.close()

# Step 9: Save Chart.js for Presentation
chartjs_config = {
    "type": "bar",
    "data": {
        "labels": ["Paddy", "Maize", "Wheat"],
        "datasets": [
            {"label": "RMSE (t/ha)", "data": rmse_values, "backgroundColor": "#1f77b4"},
            {"label": "R²", "data": r2_values, "backgroundColor": "#ff7f0e"}
        ]
    },
    "options": {
        "scales": {
            "y": {"beginAtZero": True, "title": {"display": True, "text": "Value"}},
            "x": {"title": {"display": True, "text": "Crop"}}
        },
        "plugins": {"title": {"display": True, "text": "Model Performance: RMSE and R²"}}
    }
}
with open('model_performance_chart.json', 'w') as f:
    import json
    json.dump(chartjs_config, f, indent=2)

# Save processed data
data.to_csv('processed_data.csv', index=False)

Paddy_Yield:
  RMSE: 0.280 t/ha
  R2: 0.896
  Cross-Validation R2: 0.829 (±0.062)
  Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Maize_Yield:
  RMSE: 0.355 t/ha
  R2: 0.740
  Cross-Validation R2: 0.831 (±0.063)
  Best Parameters: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
Wheat_Yield:
  RMSE: 0.267 t/ha
  R2: 0.871
  Cross-Validation R2: 0.830 (±0.063)
  Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import re
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Step 1: Load and Combine Agricultural Data
files = [
    '2013-2014.csv', '2014-2015.csv', '2015-2016.csv', '2016-2017.csv', 
    '2017-2018.csv', '2018-2019.csv', '2019-2020.csv', '2020-2021.csv', 
    '2021-2022.csv', '2022-2023.csv', '2023-2024.csv'
]

# Column name mapping to standardize variations
column_mapping = {
    r'(?i)^Paddy\s*(Area|area)\s*\(?[hH]a\)?$': 'Paddy_Area_ha',
    r'(?i)^Paddy\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Paddy_Production_MT',
    r'(?i)^Paddy\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Paddy_Yield',
    r'(?i)^Maize\s*(Area|area)\s*\(?[hH]a\)?$': 'Maize_Area_ha',
    r'(?i)^Maize\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Maize_Production_MT',
    r'(?i)^Maize\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Maize_Yield',
    r'(?i)^Wheat\s*(Area|area)\s*\(?[hH]a\)?$': 'Wheat_Area_ha',
    r'(?i)^Wheat\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Wheat_Production_MT',
    r'(?i)^Wheat\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Wheat_Yield',
    r'(?i)^District$': 'District',
    r'(?i)^Year$': 'Year'
}

dfs = []
for file in files:
    df = pd.read_csv(file)
    # Standardize column names using regex mapping
    new_columns = {}
    for col in df.columns:
        for pattern, standard_name in column_mapping.items():
            if re.match(pattern, col):
                new_columns[col] = standard_name
                break
        else:
            new_columns[col] = col
    df.rename(columns=new_columns, inplace=True)
    
    # Add Year if missing
    if 'Year' not in df.columns:
        df['Year'] = file.split('.')[0]
    
    # Standardize district names
    df['District'] = df['District'].str.upper().replace({
        'KAVREPALANCHOK': 'KAVRE',
        'NAWALPARASI_EAST': 'NAWALPUR_EAST',
        'NAWALPARASI_WEST': 'NAWALPUR_WEST',
        'EASTERN_RUKUM': 'RUKUM_EAST',
        'WESTERN_RUKUM': 'RUKUM_WEST',
        'NAWALPARASI_(WEST_OF_BARDAGHAT_SUSTA)': 'NAWALPUR_WEST',
        'SANKHUWASAVA': 'SANKHUWASABHA'
    })
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

# Calculate yields for files missing them
for crop in ['Paddy', 'Maize', 'Wheat']:
    if f'{crop}_Yield' not in data.columns or data[f'{crop}_Yield'].isna().any():
        if f'{crop}_Production_MT' in data.columns and f'{crop}_Area_ha' in data.columns:
            data[f'{crop}_Yield'] = data[f'{crop}_Production_MT'] / data[f'{crop}_Area_ha'].replace(0, np.nan)
        else:
            print(f"Warning: Missing {crop}_Production_MT or {crop}_Area_ha in some files")

# Handle missing values and zeros
data.replace(0, np.nan, inplace=True)
data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']] = data.groupby('District')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].transform(lambda x: x.fillna(x.median()))
data.dropna(subset=['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield'], inplace=True)

# Step 2: Merge Climate Data
climate_data = pd.read_csv('Nepal_77districts_annual_climate.csv')
climate_data['District'] = climate_data['District'].str.upper().replace({'SANKHUWASAVA': 'SANKHUWASABHA'})
climate_data['Year'] = climate_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')
data = pd.merge(data, climate_data, on=['Year', 'District'], how='inner')

# Step 3: Distribute Fertilizer Data
fertilizer_data = pd.read_csv('fertilizer_annual_2013_2024.csv')
fertilizer_data['Year'] = fertilizer_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')
data['Total_Area'] = data[['Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha']].sum(axis=1)
data = data.merge(data.groupby('Year')['Total_Area'].sum().reset_index(name='National_Total_Area'), on='Year')
data['Area_Share'] = data['Total_Area'] / data['National_Total_Area']
data = pd.merge(data, fertilizer_data, on='Year', how='left')
for fert in ['Urea', 'DAP', 'Potash']:
    data[f'District_{fert}_MT'] = data[fert] * data['Area_Share']

# Step 4: EDA
# Yield trends
national_yields = data.groupby('Year')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].mean().reset_index()
plt.figure(figsize=(10, 6))
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.plot(national_yields['Year'], national_yields[crop], label=crop)
plt.xlabel('Year')
plt.ylabel('Yield (t/ha)')
plt.title('National Crop Yield Trends (2013-2024)')
plt.legend()
plt.xticks(rotation=45)
plt.savefig('yield_trends.png')
plt.close()

# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield', 'Annual_Mean_Temp', 
                  'Annual_Total_Precip', 'Monsoon_Total_Precip', 'District_Urea_MT', 
                  'District_DAP_MT', 'District_Potash_MT']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Between Features and Crop Yields')
plt.savefig('correlation_heatmap.png')
plt.close()

# Yield distributions
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.figure(figsize=(8, 6))
    sns.histplot(data[crop], bins=30, kde=True)
    plt.xlabel(f'{crop} (t/ha)')
    plt.title(f'Distribution of {crop}')
    plt.savefig(f'{crop}_distribution.png')
    plt.close()

# Step 5: Feature Selection
features = ['Annual_Mean_Temp', 'Annual_Total_Precip', 'Monsoon_Total_Precip', 
            'District_Urea_MT', 'District_DAP_MT', 'District_Potash_MT', 
            'Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha', 'Latitude', 'Longitude']
scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])

# Step 6: Model Development
target_crops = ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']
results = {}

for crop in target_crops:
    X = data[features]
    y = data[crop]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }
    grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    results[crop] = {
        'Best_Model': grid_search.best_estimator_, 
        'X_test': X_test, 
        'y_test': y_test,
        'Best_Params': grid_search.best_params_
    }

# Step 7: Model Evaluation
rmse_values = []
r2_values = []
for crop in target_crops:
    model = results[crop]['Best_Model']
    y_pred = model.predict(results[crop]['X_test'])
    rmse = np.sqrt(mean_squared_error(results[crop]['y_test'], y_pred))
    r2 = r2_score(results[crop]['y_test'], y_pred)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    
    rmse_values.append(rmse)
    r2_values.append(r2)
    
    print(f"{crop}:")
    print(f"  RMSE: {rmse:.3f} t/ha")
    print(f"  R2: {r2:.3f}")
    print(f"  Cross-Validation R2: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
    print(f"  Best Parameters: {results[crop]['Best_Params']}")
    
    # Feature importance
    importance = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_})
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Importance', y='Feature', data=importance.sort_values(by='Importance', ascending=False))
    plt.title(f'Feature Importance for {crop}')
    plt.savefig(f'feature_importance_{crop}.png')
    plt.close()

# Step 8: Scenario Analysis
paddy_model = results['Paddy_Yield']['Best_Model']
X_scenario = results['Paddy_Yield']['X_test'].copy()
scenarios = {
    'Baseline': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0},
    'Rainfall_+10%': {'Annual_Total_Precip': 0.1, 'District_Urea_MT': 0},
    'Rainfall_-10%': {'Annual_Total_Precip': -0.1, 'District_Urea_MT': 0},
    'Urea_+10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0.1},
    'Urea_-10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': -0.1}
}
scenario_results = {}

for scenario, changes in scenarios.items():
    X_temp = X_scenario.copy()
    for feature, change in changes.items():
        X_temp[feature] = X_scenario[feature] * (1 + change)
    scenario_results[scenario] = paddy_model.predict(X_temp).mean()

plt.figure(figsize=(8, 6))
plt.bar(scenario_results.keys(), scenario_results.values())
plt.ylabel('Predicted Paddy Yield (t/ha)')
plt.title('Scenario Analysis: Rainfall and Urea Impact')
plt.xticks(rotation=45)
plt.savefig('scenario_analysis.png')
plt.close()

# Step 9: District-Specific Yield Predictions for 2023-2024
data_2023_2024 = data[data['Year'] == '2023-2024'].copy()
if len(data_2023_2024) != 77:
    print(f"Warning: Expected 77 districts in 2023-2024, found {len(data_2023_2024)}")

# Predict yields for each crop
for crop in target_crops:
    model = results[crop]['Best_Model']
    data_2023_2024[f'{crop}_Predicted'] = model.predict(data_2023_2024[features])

# Save predictions
prediction_columns = ['District', 'Year', 
                     'Paddy_Yield', 'Paddy_Yield_Predicted', 
                     'Maize_Yield', 'Maize_Yield_Predicted', 
                     'Wheat_Yield', 'Wheat_Yield_Predicted']
data_2023_2024[prediction_columns].to_csv('district_predictions_2023_2024.csv', index=False)

# Visualize top 10 districts by predicted yield
for crop in target_crops:
    top_districts = data_2023_2024[['District', f'{crop}_Predicted']].sort_values(by=f'{crop}_Predicted', ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=f'{crop}_Predicted', y='District', data=top_districts)
    plt.xlabel(f'Predicted {crop} (t/ha)')
    plt.title(f'Top 10 Districts by Predicted {crop} (2023-2024)')
    plt.savefig(f'district_yield_predictions_{crop}.png')
    plt.close()

# Step 10: Save Chart.js for Presentation
chartjs_config = {
    "type": "bar",
    "data": {
        "labels": ["Paddy", "Maize", "Wheat"],
        "datasets": [
            {"label": "RMSE (t/ha)", "data": rmse_values, "backgroundColor": "#1f77b4"},
            {"label": "R²", "data": r2_values, "backgroundColor": "#ff7f0e"}
        ]
    },
    "options": {
        "scales": {
            "y": {"beginAtZero": True, "title": {"display": True, "text": "Value"}},
            "x": {"title": {"display": True, "text": "Crop"}}
        },
        "plugins": {"title": {"display": True, "text": "Model Performance: RMSE and R²"}}
    }
}
with open('model_performance_chart.json', 'w') as f:
    import json
    json.dump(chartjs_config, f, indent=2)

# Save processed data
data.to_csv('processed_data.csv', index=False)

Paddy_Yield:
  RMSE: 0.268 t/ha
  R2: 0.911
  Cross-Validation R2: 0.759 (±0.173)
  Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Maize_Yield:
  RMSE: 0.382 t/ha
  R2: 0.742
  Cross-Validation R2: 0.757 (±0.180)
  Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Wheat_Yield:
  RMSE: 0.273 t/ha
  R2: 0.859
  Cross-Validation R2: 0.759 (±0.172)
  Best Parameters: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import re
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Step 1: Load and Combine Agricultural Data
files = [
    '2013-2014.csv', '2014-2015.csv', '2015-2016.csv', '2016-2017.csv', 
    '2017-2018.csv', '2018-2019.csv', '2019-2020.csv', '2020-2021.csv', 
    '2021-2022.csv', '2022-2023.csv', '2023-2024.csv'
]

# Column name mapping to standardize variations
column_mapping = {
    r'(?i)^Paddy\s*(Area|area)\s*\(?[hH]a\)?$': 'Paddy_Area_ha',
    r'(?i)^Paddy\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Paddy_Production_MT',
    r'(?i)^Paddy\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Paddy_Yield',
    r'(?i)^Maize\s*(Area|area)\s*\(?[hH]a\)?$': 'Maize_Area_ha',
    r'(?i)^Maize\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Maize_Production_MT',
    r'(?i)^Maize\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Maize_Yield',
    r'(?i)^Wheat\s*(Area|area)\s*\(?[hH]a\)?$': 'Wheat_Area_ha',
    r'(?i)^Wheat\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Wheat_Production_MT',
    r'(?i)^Wheat\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Wheat_Yield',
    r'(?i)^District$': 'District',
    r'(?i)^Year$': 'Year'
}

dfs = []
for file in files:
    df = pd.read_csv(file)
    # Standardize column names using regex mapping
    new_columns = {}
    for col in df.columns:
        for pattern, standard_name in column_mapping.items():
            if re.match(pattern, col):
                new_columns[col] = standard_name
                break
        else:
            new_columns[col] = col
    df.rename(columns=new_columns, inplace=True)
    
    # Add Year if missing
    if 'Year' not in df.columns:
        df['Year'] = file.split('.')[0]
    
    # Standardize district names
    df['District'] = df['District'].str.upper().replace({
        'KAVREPALANCHOK': 'KAVRE',
        'NAWALPARASI_EAST': 'NAWALPUR_EAST',
        'NAWALPARASI_WEST': 'NAWALPUR_WEST',
        'EASTERN_RUKUM': 'RUKUM_EAST',
        'WESTERN_RUKUM': 'RUKUM_WEST',
        'NAWALPARASI_(WEST_OF_BARDAGHAT_SUSTA)': 'NAWALPUR_WEST',
        'SANKHUWASAVA': 'SANKHUWASABHA'
    })
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

# Calculate yields for files missing them
for crop in ['Paddy', 'Maize', 'Wheat']:
    if f'{crop}_Yield' not in data.columns or data[f'{crop}_Yield'].isna().any():
        if f'{crop}_Production_MT' in data.columns and f'{crop}_Area_ha' in data.columns:
            data[f'{crop}_Yield'] = data[f'{crop}_Production_MT'] / data[f'{crop}_Area_ha'].replace(0, np.nan)
        else:
            print(f"Warning: Missing {crop}_Production_MT or {crop}_Area_ha in some files")

# Handle missing values and zeros
data.replace(0, np.nan, inplace=True)
data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']] = data.groupby('District')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].transform(lambda x: x.fillna(x.median()))
data.dropna(subset=['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield'], inplace=True)

# Step 2: Merge Climate Data
climate_data = pd.read_csv('Nepal_77districts_annual_climate.csv')
climate_data['District'] = climate_data['District'].str.upper().replace({'SANKHUWASAVA': 'SANKHUWASABHA'})
climate_data['Year'] = climate_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')

# Use left merge to retain all agricultural data
data = pd.merge(data, climate_data, on=['Year', 'District'], how='left')
# Impute missing climate data with median values
for col in ['Annual_Mean_Temp', 'Annual_Total_Precip', 'Monsoon_Total_Precip', 'Latitude', 'Longitude']:
    data[col] = data[col].fillna(data[col].median())

# Step 3: Distribute Fertilizer Data
fertilizer_data = pd.read_csv('fertilizer_annual_2013_2024.csv')
fertilizer_data['Year'] = fertilizer_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')
data['Total_Area'] = data[['Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha']].sum(axis=1)
data = data.merge(data.groupby('Year')['Total_Area'].sum().reset_index(name='National_Total_Area'), on='Year')
data['Area_Share'] = data['Total_Area'] / data['National_Total_Area']
data = pd.merge(data, fertilizer_data, on='Year', how='left')
for fert in ['Urea', 'DAP', 'Potash']:
    data[f'District_{fert}_MT'] = data[fert] * data['Area_Share']
# Impute missing fertilizer data
for col in ['District_Urea_MT', 'District_DAP_MT', 'District_Potash_MT']:
    data[col] = data[col].fillna(data[col].median())

# Step 4: Check for Missing Districts in 2023-2024
data_2023_2024 = data[data['Year'] == '2023-2024'].copy()
if len(data_2023_2024['District'].unique()) != 77:
    print(f"Warning: Expected 77 districts in 2023-2024, found {len(data_2023_2024['District'].unique())}")
    missing_districts = set(climate_data[climate_data['Year'] == '2023-2024']['District'].unique()) - set(data_2023_2024['District'].unique())
    print(f"Missing districts: {missing_districts}")
    # Impute missing districts using median values from historical data
    for district in missing_districts:
        historical_data = data[data['District'] == district]
        if not historical_data.empty:
            new_row = historical_data.median(numeric_only=True).to_dict()
            new_row['District'] = district
            new_row['Year'] = '2023-2024'
            data_2023_2024 = pd.concat([data_2023_2024, pd.DataFrame([new_row])], ignore_index=True)

# Step 5: EDA
# Yield trends
national_yields = data.groupby('Year')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].mean().reset_index()
plt.figure(figsize=(10, 6))
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.plot(national_yields['Year'], national_yields[crop], label=crop)
plt.xlabel('Year')
plt.ylabel('Yield (t/ha)')
plt.title('National Crop Yield Trends (2013-2024)')
plt.legend()
plt.xticks(rotation=45)
plt.savefig('yield_trends.png')
plt.close()

# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield', 'Annual_Mean_Temp', 
                  'Annual_Total_Precip', 'Monsoon_Total_Precip', 'District_Urea_MT', 
                  'District_DAP_MT', 'District_Potash_MT']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Between Features and Crop Yields')
plt.savefig('correlation_heatmap.png')
plt.close()

# Yield distributions
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.figure(figsize=(8, 6))
    sns.histplot(data[crop], bins=30, kde=True)
    plt.xlabel(f'{crop} (t/ha)')
    plt.title(f'Distribution of {crop}')
    plt.savefig(f'{crop}_distribution.png')
    plt.close()

# Step 6: Feature Selection
features = ['Annual_Mean_Temp', 'Annual_Total_Precip', 'Monsoon_Total_Precip', 
            'District_Urea_MT', 'District_DAP_MT', 'District_Potash_MT', 
            'Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha', 'Latitude', 'Longitude']
scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])
data_2023_2024[features] = scaler.transform(data_2023_2024[features])

# Step 7: Model Development
target_crops = ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']
results = {}

for crop in target_crops:
    X = data[features]
    y = data[crop]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }
    grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    results[crop] = {
        'Best_Model': grid_search.best_estimator_, 
        'X_test': X_test, 
        'y_test': y_test,
        'Best_Params': grid_search.best_params_
    }

# Step 8: Model Evaluation
rmse_values = []
r2_values = []
for crop in target_crops:
    model = results[crop]['Best_Model']
    y_pred = model.predict(results[crop]['X_test'])
    rmse = np.sqrt(mean_squared_error(results[crop]['y_test'], y_pred))
    r2 = r2_score(results[crop]['y_test'], y_pred)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    
    rmse_values.append(rmse)
    r2_values.append(r2)
    
    print(f"{crop}:")
    print(f"  RMSE: {rmse:.3f} t/ha")
    print(f"  R2: {r2:.3f}")
    print(f"  Cross-Validation R2: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
    print(f"  Best Parameters: {results[crop]['Best_Params']}")
    
    # Feature importance
    importance = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_})
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Importance', y='Feature', data=importance.sort_values(by='Importance', ascending=False))
    plt.title(f'Feature Importance for {crop}')
    plt.savefig(f'feature_importance_{crop}.png')
    plt.close()

# Step 9: Scenario Analysis
paddy_model = results['Paddy_Yield']['Best_Model']
X_scenario = results['Paddy_Yield']['X_test'].copy()
scenarios = {
    'Baseline': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0},
    'Rainfall_+10%': {'Annual_Total_Precip': 0.1, 'District_Urea_MT': 0},
    'Rainfall_-10%': {'Annual_Total_Precip': -0.1, 'District_Urea_MT': 0},
    'Urea_+10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0.1},
    'Urea_-10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': -0.1}
}
scenario_results = {}

for scenario, changes in scenarios.items():
    X_temp = X_scenario.copy()
    for feature, change in changes.items():
        X_temp[feature] = X_scenario[feature] * (1 + change)
    scenario_results[scenario] = paddy_model.predict(X_temp).mean()

plt.figure(figsize=(8, 6))
plt.bar(scenario_results.keys(), scenario_results.values())
plt.ylabel('Predicted Paddy Yield (t/ha)')
plt.title('Scenario Analysis: Rainfall and Urea Impact')
plt.xticks(rotation=45)
plt.savefig('scenario_analysis.png')
plt.close()

# Step 10: District-Specific Yield Predictions for 2023-2024
# Predict yields for each crop
for crop in target_crops:
    model = results[crop]['Best_Model']
    data_2023_2024[f'{crop}_Predicted'] = model.predict(data_2023_2024[features])
    data_2023_2024[f'{crop}_Error'] = data_2023_2024[f'{crop}_Predicted'] - data_2023_2024[f'{crop}']

# Save predictions
prediction_columns = ['District', 'Year', 
                     'Paddy_Yield', 'Paddy_Yield_Predicted', 'Paddy_Yield_Error',
                     'Maize_Yield', 'Maize_Yield_Predicted', 'Maize_Yield_Error',
                     'Wheat_Yield', 'Wheat_Yield_Predicted', 'Wheat_Yield_Error']
data_2023_2024[prediction_columns].to_csv('district_predictions_2023_2024.csv', index=False)

# Visualize top 10 districts by predicted yield
for crop in target_crops:
    top_districts = data_2023_2024[['District', f'{crop}_Predicted']].sort_values(by=f'{crop}_Predicted', ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=f'{crop}_Predicted', y='District', data=top_districts)
    plt.xlabel(f'Predicted {crop} (t/ha)')
    plt.title(f'Top 10 Districts by Predicted {crop} (2023-2024)')
    plt.savefig(f'district_yield_predictions_{crop}.png')
    plt.close()

# Visualize top 10 districts by prediction error
for crop in target_crops:
    top_errors = data_2023_2024[['District', f'{crop}_Error']].sort_values(by=f'{crop}_Error', key=abs, ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=f'{crop}_Error', y='District', data=top_errors)
    plt.xlabel(f'Prediction Error for {crop} (t/ha)')
    plt.title(f'Top 10 Districts by {crop} Prediction Error (2023-2024)')
    plt.savefig(f'district_yield_error_{crop}.png')
    plt.close()

# Step 11: Save Chart.js for Presentation
chartjs_config = {
    "type": "bar",
    "data": {
        "labels": ["Paddy", "Maize", "Wheat"],
        "datasets": [
            {"label": "RMSE (t/ha)", "data": [0.268, 0.382, 0.273], "backgroundColor": "#1f77b4"},
            {"label": "R²", "data": [0.911, 0.742, 0.859], "backgroundColor": "#ff7f0e"}
        ]
    },
    "options": {
        "scales": {
            "y": {"beginAtZero": True, "title": {"display": True, "text": "Value"}},
            "x": {"title": {"display": True, "text": "Crop"}}
        },
        "plugins": {"title": {"display": True, "text": "Model Performance: RMSE and R²"}}
    }
}
with open('model_performance_chart.json', 'w') as f:
    import json
    json.dump(chartjs_config, f, indent=2)

# Save processed data
data.to_csv('processed_data.csv', index=False)

Missing districts: {'RAMECHHAP', 'ILAM', 'KAVREPALANCHOK', 'NAWALPARASI BARDAGHAT SUSTA WEST', 'DHANUSA', 'MANANG', 'MUSTANG', 'SANKHUWASABHA', 'CHITAWAN', 'NAWALPARASI BARDAGHAT SUSTA EAST'}
Paddy_Yield:
  RMSE: 0.253 t/ha
  R2: 0.897
  Cross-Validation R2: 0.758 (±0.123)
  Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Maize_Yield:
  RMSE: 0.327 t/ha
  R2: 0.766
  Cross-Validation R2: 0.758 (±0.124)
  Best Parameters: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
Wheat_Yield:
  RMSE: 0.266 t/ha
  R2: 0.832
  Cross-Validation R2: 0.758 (±0.123)
  Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import re
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Step 1: Load and Combine Agricultural Data
files = [
    '2013-2014.csv', '2014-2015.csv', '2015-2016.csv', '2016-2017.csv', 
    '2017-2018.csv', '2018-2019.csv', '2019-2020.csv', '2020-2021.csv', 
    '2021-2022.csv', '2022-2023.csv', '2023-2024.csv'
]

# Column name mapping to standardize variations
column_mapping = {
    r'(?i)^Paddy\s*(Area|area)\s*\(?[hH]a\)?$': 'Paddy_Area_ha',
    r'(?i)^Paddy\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Paddy_Production_MT',
    r'(?i)^Paddy\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Paddy_Yield',
    r'(?i)^Maize\s*(Area|area)\s*\(?[hH]a\)?$': 'Maize_Area_ha',
    r'(?i)^Maize\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Maize_Production_MT',
    r'(?i)^Maize\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Maize_Yield',
    r'(?i)^Wheat\s*(Area|area)\s*\(?[hH]a\)?$': 'Wheat_Area_ha',
    r'(?i)^Wheat\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Wheat_Production_MT',
    r'(?i)^Wheat\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Wheat_Yield',
    r'(?i)^District$': 'District',
    r'(?i)^Year$': 'Year'
}

# Expanded district name standardization
district_mapping = {
    'KAVREPALANCHOK': 'KAVRE',
    'NAWALPARASI_EAST': 'NAWALPUR_EAST',
    'NAWALPARASI_WEST': 'NAWALPUR_WEST',
    'EASTERN_RUKUM': 'RUKUM_EAST',
    'WESTERN_RUKUM': 'RUKUM_WEST',
    'NAWALPARASI_(WEST_OF_BARDAGHAT_SUSTA)': 'NAWALPUR_WEST',
    'NAWALPARASI BARDAGHAT SUSTA WEST': 'NAWALPUR_WEST',
    'NAWALPARASI BARDAGHAT SUSTA EAST': 'NAWALPUR_EAST',
    'SANKHUWASAVA': 'SANKHUWASABHA',
    'SANKHUWASABHA': 'SANKHUWASABHA',
    'CHITAWAN': 'CHITWAN',
    'RAMECHHAP': 'RAMECHHAP',
    'ILAM': 'ILAM',
    'DHANUSA': 'DHANUSHA',
    'MANANG': 'MANANG',
    'MUSTANG': 'MUSTANG'
}

dfs = []
for file in files:
    df = pd.read_csv(file)
    # Standardize column names
    new_columns = {}
    for col in df.columns:
        for pattern, standard_name in column_mapping.items():
            if re.match(pattern, col):
                new_columns[col] = standard_name
                break
        else:
            new_columns[col] = col
    df.rename(columns=new_columns, inplace=True)
    
    # Add Year if missing
    if 'Year' not in df.columns:
        df['Year'] = file.split('.')[0]
    
    # Standardize district names
    df['District'] = df['District'].str.upper().replace(district_mapping)
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

# Debug: Print unique districts in 2023-2024 agricultural data
agri_2023_2024 = data[data['Year'] == '2023-2024']
print(f"Unique districts in 2023-2024 agricultural data: {len(agri_2023_2024['District'].unique())}")
print(f"Districts: {sorted(agri_2023_2024['District'].unique())}")

# Calculate yields for files missing them
for crop in ['Paddy', 'Maize', 'Wheat']:
    if f'{crop}_Yield' not in data.columns or data[f'{crop}_Yield'].isna().any():
        if f'{crop}_Production_MT' in data.columns and f'{crop}_Area_ha' in data.columns:
            data[f'{crop}_Yield'] = data[f'{crop}_Production_MT'] / data[f'{crop}_Area_ha'].replace(0, np.nan)
        else:
            print(f"Warning: Missing {crop}_Production_MT or {crop}_Area_ha in some files")

# Handle missing values and zeros
data.replace(0, np.nan, inplace=True)
data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']] = data.groupby('District')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].transform(lambda x: x.fillna(x.median()))
data.dropna(subset=['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield'], inplace=True)

# Step 2: Merge Climate Data
climate_data = pd.read_csv('Nepal_77districts_annual_climate.csv')
climate_data['District'] = climate_data['District'].str.upper().replace(district_mapping)
climate_data['Year'] = climate_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')

# Debug: Print unique districts in climate data
climate_2023_2024 = climate_data[climate_data['Year'] == '2023-2024']
print(f"Unique districts in 2023-2024 climate data: {len(climate_2023_2024['District'].unique())}")
print(f"Districts: {sorted(climate_2023_2024['District'].unique())}")

# Use left merge to retain all agricultural data
data = pd.merge(data, climate_data, on=['Year', 'District'], how='left')
# Impute missing climate data
for col in ['Annual_Mean_Temp', 'Annual_Total_Precip', 'Monsoon_Total_Precip', 'Latitude', 'Longitude']:
    data[col] = data[col].fillna(data[col].median())

# Step 3: Distribute Fertilizer Data
fertilizer_data = pd.read_csv('fertilizer_annual_2013_2024.csv')
fertilizer_data['Year'] = fertilizer_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')
data['Total_Area'] = data[['Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha']].sum(axis=1)
data = data.merge(data.groupby('Year')['Total_Area'].sum().reset_index(name='National_Total_Area'), on='Year')
data['Area_Share'] = data['Total_Area'] / data['National_Total_Area']
data = pd.merge(data, fertilizer_data, on='Year', how='left')
for fert in ['Urea', 'DAP', 'Potash']:
    data[f'District_{fert}_MT'] = data[fert] * data['Area_Share']
# Impute missing fertilizer data
for col in ['District_Urea_MT', 'District_DAP_MT', 'District_Potash_MT']:
    data[col] = data[col].fillna(data[col].median())

# Step 4: Check for Missing Districts in 2023-2024
data_2023_2024 = data[data['Year'] == '2023-2024'].copy()
if len(data_2023_2024['District'].unique()) != 77:
    print(f"Warning: Expected 77 districts in 2023-2024, found {len(data_2023_2024['District'].unique())}")
    missing_districts = set(climate_data[climate_data['Year'] == '2023-2024']['District'].unique()) - set(data_2023_2024['District'].unique())
    print(f"Missing districts: {missing_districts}")
    # Impute missing districts
    for district in missing_districts:
        historical_data = data[data['District'] == district]
        if not historical_data.empty:
            new_row = historical_data.median(numeric_only=True).to_dict()
            new_row['District'] = district
            new_row['Year'] = '2023-2024'
            data_2023_2024 = pd.concat([data_2023_2024, pd.DataFrame([new_row])], ignore_index=True)
    print(f"After imputation, districts in 2023-2024: {len(data_2023_2024['District'].unique())}")

# Step 5: EDA
# Yield trends
national_yields = data.groupby('Year')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].mean().reset_index()
plt.figure(figsize=(10, 6))
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.plot(national_yields['Year'], national_yields[crop], label=crop)
plt.xlabel('Year')
plt.ylabel('Yield (t/ha)')
plt.title('National Crop Yield Trends (2013-2024)')
plt.legend()
plt.xticks(rotation=45)
plt.savefig('yield_trends.png')
plt.close()

# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield', 'Annual_Mean_Temp', 
                  'Annual_Total_Precip', 'Monsoon_Total_Precip', 'District_Urea_MT', 
                  'District_DAP_MT', 'District_Potash_MT']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Between Features and Crop Yields')
plt.savefig('correlation_heatmap.png')
plt.close()

# Yield distributions
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.figure(figsize=(8, 6))
    sns.histplot(data[crop], bins=30, kde=True)
    plt.xlabel(f'{crop} (t/ha)')
    plt.title(f'Distribution of {crop}')
    plt.savefig(f'{crop}_distribution.png')
    plt.close()

# Step 6: Feature Selection
features = ['Annual_Mean_Temp', 'Annual_Total_Precip', 'Monsoon_Total_Precip', 
            'District_Urea_MT', 'District_DAP_MT', 'District_Potash_MT', 
            'Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha', 'Latitude', 'Longitude']
scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])
data_2023_2024[features] = scaler.transform(data_2023_2024[features])

# Step 7: Model Development
target_crops = ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']
results = {}

for crop in target_crops:
    X = data[features]
    y = data[crop]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }
    grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    results[crop] = {
        'Best_Model': grid_search.best_estimator_, 
        'X_test': X_test, 
        'y_test': y_test,
        'Best_Params': grid_search.best_params_
    }

# Step 8: Model Evaluation
rmse_values = []
r2_values = []
for crop in target_crops:
    model = results[crop]['Best_Model']
    y_pred = model.predict(results[crop]['X_test'])
    rmse = np.sqrt(mean_squared_error(results[crop]['y_test'], y_pred))
    r2 = r2_score(results[crop]['y_test'], y_pred)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    
    rmse_values.append(rmse)
    r2_values.append(r2)
    
    print(f"{crop}:")
    print(f"  RMSE: {rmse:.3f} t/ha")
    print(f"  R2: {r2:.3f}")
    print(f"  Cross-Validation R2: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
    print(f"  Best Parameters: {results[crop]['Best_Params']}")
    
    # Feature importance
    importance = pd.DataFrame({'Feature': findings, 'Importance': model.feature_importances_})
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Importance', y='Feature', data=importance.sort_values(by='Importance', ascending=False))
    plt.title(f'Feature Importance for {crop}')
    plt.savefig(f'feature_importance_{crop}.png')
    plt.close()

# Step 9: Scenario Analysis
paddy_model = results['Paddy_Yield']['Best_Model']
X_scenario = results['Paddy_Yield']['X_test'].copy()
scenarios = {
    'Baseline': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0},
    'Rainfall_+10%': {'Annual_Total_Precip': 0.1, 'District_Urea_MT': 0},
    'Rainfall_-10%': {'Annual_Total_Precip': -0.1, 'District_Urea_MT': 0},
    'Urea_+10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0.1},
    'Urea_-10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': -0.1}
}
scenario_results = {}

for scenario, changes in scenarios.items():
    X_temp = X_scenario.copy()
    for feature, change in changes.items():
        X_temp[feature] = X_scenario[feature] * (1 + change)
    scenario_results[scenario] = paddy_model.predict(X_temp).mean()

plt.figure(figsize=(8, 6))
plt.bar(scenario_results.keys(), scenario_results.values())
plt.ylabel('Predicted Paddy Yield (t/ha)')
plt.title('Scenario Analysis: Rainfall and Urea Impact')
plt.xticks(rotation=45)
plt.savefig('scenario_analysis.png')
plt.close()

# Step 10: District-Specific Yield Predictions for 2023-2024
# Predict yields for each crop
for crop in target_crops:
    model = results[crop]['Best_Model']
    data_2023_2024[f'{crop}_Predicted'] = model.predict(data_2023_2024[features])
    data_2023_2024[f'{crop}_Error'] = data_2023_2024[f'{crop}_Predicted'] - data_2023_2024[f'{crop}']

# Save predictions
prediction_columns = ['District', 'Year', 
                     'Paddy_Yield', 'Paddy_Yield_Predicted', 'Paddy_Yield_Error',
                     'Maize_Yield', 'Maize_Yield_Predicted', 'Maize_Yield_Error',
                     'Wheat_Yield', 'Wheat_Yield_Predicted', 'Wheat_Yield_Error']
data_2023_2024[prediction_columns].to_csv('district_predictions_2023_2024.csv', index=False)

# Visualize top 10 districts by predicted yield
for crop in target_crops:
    top_districts = data_2023_2024[['District', f'{crop}_Predicted']].sort_values(by=f'{crop}_Predicted', ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=f'{crop}_Predicted', y='District', data=top_districts)
    plt.xlabel(f'Predicted {crop} (t/ha)')
    plt.title(f'Top 10 Districts by Predicted {crop} (2023-2024)')
    plt.savefig(f'district_yield_predictions_{crop}.png')
    plt.close()

# Visualize top 10 districts by prediction error
for crop in target_crops:
    top_errors = data_2023_2024[['District', f'{crop}_Error']].sort_values(by=f'{crop}_Error', key=abs, ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=f'{crop}_Error', y='District', data=top_errors)
    plt.xlabel(f'Prediction Error for {crop} (t/ha)')
    plt.title(f'Top 10 Districts by {crop} Prediction Error (2023-2024)')
    plt.savefig(f'district_yield_error_{crop}.png')
    plt.close()

# Step 11: Save Chart.js for Presentation
chartjs_config = {
    "type": "bar",
    "data": {
        "labels": ["Paddy", "Maize", "Wheat"],
        "datasets": [
            {"label": "RMSE (t/ha)", "data": [0.268, 0.382, 0.273], "backgroundColor": "#1f77b4"},
            {"label": "R²", "data": [0.911, 0.742, 0.859], "backgroundColor": "#ff7f0e"}
        ]
    },
    "options": {
        "scales": {
            "y": {"beginAtZero": True, "title": {"display": True, "text": "Value"}},
            "x": {"title": {"display": True, "text": "Crop"}}
        },
        "plugins": {"title": {"display": True, "text": "Model Performance: RMSE and R²"}}
    }
}
with open('model_performance_chart.json', 'w') as f:
    import json
    json.dump(chartjs_config, f, indent=2)

# Save processed data
data.to_csv('processed_data.csv', index=False)

Unique districts in 2023-2024 agricultural data: 77
Districts: ['ACHHAM', 'ARGHAKHANCHI', 'BAGLUNG', 'BAITADI', 'BAJHANG', 'BAJURA', 'BANKE', 'BARA', 'BARDIYA', 'BHAKTAPUR', 'BHOJPUR', 'CHITWAN', 'DADELDHURA', 'DAILEKH', 'DANG', 'DARCHULA', 'DHADING', 'DHANKUTA', 'DHANUSHA', 'DOLAKHA', 'DOLPA', 'DOTI', 'GORKHA', 'GULMI', 'HUMLA', 'ILLAM', 'JAJARKOT', 'JHAPA', 'JUMLA', 'KAILALI', 'KALIKOT', 'KANCHANPUR', 'KAPILBASTU', 'KASKI', 'KATHMANDU', 'KAVRE', 'KHOTANG', 'LALITPUR', 'LAMJUNG', 'MAHOTTARI', 'MAKWANPUR', 'MANANG', 'MORANG', 'MUGU', 'MUSTANG', 'MYAGDI', 'NAWALPARASI EAST', 'NAWALPARASI WEST', 'NUWAKOT', 'OKHALDHUNGA', 'PALPA', 'PANCHTHAR', 'PARBAT', 'PARSA', 'PYUTHAN', 'RAMECHAP', 'RASUWA', 'RAUTAHAT', 'ROLPA', 'RUKUM EAST', 'RUKUM WEST', 'RUPANDEHI', 'SALYAN', 'SANKHUWASHAVA', 'SAPTARI', 'SARLAHI', 'SINDHULI', 'SINDHUPALCHOK', 'SIRAHA', 'SOLUKHUMBU', 'SUNSARI', 'SURKHET', 'SYANGJA', 'TANAHU', 'TAPLEJUNG', 'TERHATHUM', 'UDAYAPUR']
Unique districts in 2023-2024 climate data: 77
Distric

NameError: name 'findings' is not defined

In [10]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import re
import warnings
import json
from collections import Counter

warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

# Base path for files
base_path = r'C:\Users\feroj\OneDrive\Desktop\specialist\\'

# Step 1: Load and Combine Agricultural Data
files = [
    '2013-2014.csv', '2014-2015.csv', '2015-2016.csv', '2016-2017.csv', 
    '2017-2018.csv', '2018-2019.csv', '2019-2020.csv', '2020-2021.csv', 
    '2021-2022.csv', '2022-2023.csv', '2023-2024.csv'
]

# Column name mapping to standardize variations
column_mapping = {
    r'(?i)^Paddy\s*(Area|area)\s*\(?[hH]a\)?$': 'Paddy_Area_ha',
    r'(?i)^Paddy\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Paddy_Production_MT',
    r'(?i)^Paddy\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Paddy_Yield',
    r'(?i)^Maize\s*(Area|area)\s*\(?[hH]a\)?$': 'Maize_Area_ha',
    r'(?i)^Maize\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Maize_Production_MT',
    r'(?i)^Maize\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Maize_Yield',
    r'(?i)^Wheat\s*(Area|area)\s*\(?[hH]a\)?$': 'Wheat_Area_ha',
    r'(?i)^Wheat\s*(Production|Prod|production)\s*\(?[mM][tT]\)?$': 'Wheat_Production_MT',
    r'(?i)^Wheat\s*(Yield|yield)\s*\(?[mM][tT]/[hH]a\)?$': 'Wheat_Yield',
    r'(?i)^District$': 'District',
    r'(?i)^Year$': 'Year'
}

# Expanded district name standardization
district_mapping = {
    'KAVREPALANCHOK': 'KAVRE',
    'NAWALPARASI': 'NAWALPUR_EAST',
    'NAWALPARASI_EAST': 'NAWALPUR_EAST',
    'NAWALPARASI_WEST': 'NAWALPUR_WEST',
    'NAWALPARASI_(WEST_OF_BARDAGHAT_SUSTA)': 'NAWALPUR_WEST',
    'NAWALPARASI BARDAGHAT SUSTA WEST': 'NAWALPUR_WEST',
    'NAWALPARASI BARDAGHAT SUSTA EAST': 'NAWALPUR_EAST',
    'EASTERN_RUKUM': 'RUKUM_EAST',
    'WESTERN_RUKUM': 'RUKUM_WEST',
    'SANKHUWASAVA': 'SANKHUWASABHA',
    'SANKHUWASHAVA': 'SANKHUWASABHA',
    'CHITAWAN': 'CHITWAN',
    'RAMECHHAP': 'RAMECHHAP',
    'DHANUSA': 'DHANUSHA',
    'MANANG': 'MANANG',
    'MUSTANG': 'MUSTANG'
}

dfs = []
for file in files:
    df = pd.read_csv(base_path + file)
    # Standardize column names
    new_columns = {}
    for col in df.columns:
        for pattern, standard_name in column_mapping.items():
            if re.match(pattern, col):
                new_columns[col] = standard_name
                break
        else:
            new_columns[col] = col
    df.rename(columns=new_columns, inplace=True)
    
    # Add Year if missing
    if 'Year' not in df.columns:
        df['Year'] = file.split('.')[0]
    
    # Standardize district names
    df['District'] = df['District'].str.upper().replace(district_mapping)
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)

# Debug: Check unique districts in 2023-2024
agri_2023_2024 = data[data['Year'] == '2023-2024']
unique_districts = agri_2023_2024['District'].unique()
print(f"Unique districts in 2023-2024 agricultural data: {len(unique_districts)}")
print(f"Districts: {sorted(unique_districts)}")

# Handle duplicates by aggregating
district_counts = Counter(data['District'])
duplicates = {district: count for district, count in district_counts.items() if count > 11}
if duplicates:
    print(f"Potential duplicates: {duplicates}")
    data = data.groupby(['District', 'Year']).mean(numeric_only=True).reset_index()

# Recheck districts after deduplication
agri_2023_2024 = data[data['Year'] == '2023-2024']
unique_districts = agri_2023_2024['District'].unique()
print(f"After deduplication, districts in 2023-2024: {len(unique_districts)}")

# Calculate yields for files missing them
for crop in ['Paddy', 'Maize', 'Wheat']:
    if f'{crop}_Yield' not in data.columns or data[f'{crop}_Yield'].isna().any():
        if f'{crop}_Production_MT' in data.columns and f'{crop}_Area_ha' in data.columns:
            data[f'{crop}_Yield'] = data[f'{crop}_Production_MT'] / data[f'{crop}_Area_ha'].replace(0, np.nan)
        else:
            print(f"Warning: Missing {crop}_Production_MT or {crop}_Area_ha in some files")

# Handle missing values and zeros
data.replace(0, np.nan, inplace=True)
data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']] = data.groupby('District')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].transform(lambda x: x.fillna(x.median()))
data.dropna(subset=['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield'], inplace=True)

# Step 2: Merge Climate Data
climate_data = pd.read_csv(base_path + 'Nepal_77districts_annual_climate.csv')
climate_data['District'] = climate_data['District'].str.upper().replace(district_mapping)
climate_data['Year'] = climate_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')

# Debug: Check climate data districts
climate_2023_2024 = climate_data[climate_data['Year'] == '2023-2024']
print(f"Unique districts in 2023-2024 climate data: {len(climate_2023_2024['District'].unique())}")
print(f"Districts: {sorted(climate_2023_2024['District'].unique())}")

# Merge with agricultural data
data = pd.merge(data, climate_data, on=['Year', 'District'], how='left')
for col in ['Annual_Mean_Temp', 'Annual_Total_Precip', 'Monsoon_Total_Precip', 'Latitude', 'Longitude']:
    data[col] = data[col].fillna(data[col].median())

# Step 3: Distribute Fertilizer Data
fertilizer_data = pd.read_csv(base_path + 'fertilizer_annual_2013_2024.csv')
fertilizer_data['Year'] = fertilizer_data['Year'].astype(str).apply(lambda x: f'{x}-{int(x)+1}')
data['Total_Area'] = data[['Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha']].sum(axis=1)
data = data.merge(data.groupby('Year')['Total_Area'].sum().reset_index(name='National_Total_Area'), on='Year')
data['Area_Share'] = data['Total_Area'] / data['National_Total_Area']
data = pd.merge(data, fertilizer_data, on='Year', how='left')
for fert in ['Urea', 'DAP', 'Potash']:
    data[f'District_{fert}_MT'] = data[fert] * data['Area_Share']
for col in ['District_Urea_MT', 'District_DAP_MT', 'District_Potash_MT']:
    data[col] = data[col].fillna(data[col].median())

# Step 4: Ensure 77 Districts in 2023-2024
data_2023_2024 = data[data['Year'] == '2023-2024'].copy()
if len(data_2023_2024['District'].unique()) != 77:
    print(f"Warning: Expected 77 districts in 2023-2024, found {len(data_2023_2024['District'].unique())}")
    missing_districts = set(climate_data[climate_data['Year'] == '2023-2024']['District'].unique()) - set(data_2023_2024['District'].unique())
    print(f"Missing districts: {missing_districts}")
    for district in missing_districts:
        historical_data = data[data['District'] == district]
        if not historical_data.empty:
            new_row = historical_data.median(numeric_only=True).to_dict()
            new_row['District'] = district
            new_row['Year'] = '2023-2024'
            data_2023_2024 = pd.concat([data_2023_2024, pd.DataFrame([new_row])], ignore_index=True)
    data = pd.concat([data[data['Year'] != '2023-2024'], data_2023_2024], ignore_index=True)
    print(f"After imputation, districts in 2023-2024: {len(data_2023_2024['District'].unique())}")

# Step 5: EDA
# Yield trends
national_yields = data.groupby('Year')[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']].mean().reset_index()
plt.figure(figsize=(10, 6))
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.plot(national_yields['Year'], national_yields[crop], label=crop)
plt.xlabel('Year')
plt.ylabel('Yield (t/ha)')
plt.title('National Crop Yield Trends (2013-2024)')
plt.legend()
plt.xticks(rotation=45)
plt.savefig(base_path + 'yield_trends.png')
plt.close()

# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data[['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield', 'Annual_Mean_Temp', 
                  'Annual_Total_Precip', 'Monsoon_Total_Precip', 'District_Urea_MT', 
                  'District_DAP_MT', 'District_Potash_MT']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Between Features and Crop Yields')
plt.savefig(base_path + 'correlation_heatmap.png')
plt.close()

# Yield distributions
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    plt.figure(figsize=(8, 6))
    sns.histplot(data[crop], bins=30, kde=True)
    plt.xlabel(f'{crop} (t/ha)')
    plt.title(f'Distribution of {crop}')
    plt.savefig(base_path + f'{crop}_distribution.png')
    plt.close()

# Step 6: Feature Selection
features = ['Annual_Mean_Temp', 'Annual_Total_Precip', 'Monsoon_Total_Precip', 
            'District_Urea_MT', 'District_DAP_MT', 'District_Potash_MT', 
            'Paddy_Area_ha', 'Maize_Area_ha', 'Wheat_Area_ha', 'Latitude', 'Longitude']
scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])
data_2023_2024[features] = scaler.transform(data_2023_2024[features])

# Step 7: Model Development
target_crops = ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']
results = {}

for crop in target_crops:
    X = data[features]
    y = data[crop]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestRegressor(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }
    grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    results[crop] = {
        'Best_Model': grid_search.best_estimator_, 
        'X_test': X_test, 
        'y_test': y_test,
        'Best_Params': grid_search.best_params_
    }

# Step 8: Model Evaluation
rmse_values = []
r2_values = []
for crop in target_crops:
    model = results[crop]['Best_Model']
    y_pred = model.predict(results[crop]['X_test'])
    rmse = np.sqrt(mean_squared_error(results[crop]['y_test'], y_pred))
    r2 = r2_score(results[crop]['y_test'], y_pred)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    
    rmse_values.append(rmse)
    r2_values.append(r2)
    
    print(f"{crop}:")
    print(f"  RMSE: {rmse:.3f} t/ha")
    print(f"  R2: {r2:.3f}")
    print(f"  Cross-Validation R2: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
    print(f"  Best Parameters: {results[crop]['Best_Params']}")
    
    # Feature importance
    importance = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_})
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Importance', y='Feature', data=importance.sort_values(by='Importance', ascending=False))
    plt.title(f'Feature Importance for {crop}')
    plt.savefig(base_path + f'feature_importance_{crop}.png')
    plt.close()
    
    # ✅ Predicted vs Actual Plot
    plt.figure(figsize=(7, 6))
    plt.scatter(results[crop]['y_test'], y_pred, alpha=0.7, color='blue')
    plt.plot([results[crop]['y_test'].min(), results[crop]['y_test'].max()],
             [results[crop]['y_test'].min(), results[crop]['y_test'].max()],
             'r--', lw=2)
    plt.xlabel("Actual Yield (t/ha)")
    plt.ylabel("Predicted Yield (t/ha)")
    plt.title(f"Predicted vs Actual Yield - {crop}")
    plt.savefig(base_path + f'predicted_vs_actual_{crop}.png')
    plt.close()


# Step 9: Scenario Analysis
paddy_model = results['Paddy_Yield']['Best_Model']
X_scenario = results['Paddy_Yield']['X_test'].copy()
scenarios = {
    'Baseline': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0},
    'Rainfall_+10%': {'Annual_Total_Precip': 0.1, 'District_Urea_MT': 0},
    'Rainfall_-10%': {'Annual_Total_Precip': -0.1, 'District_Urea_MT': 0},
    'Urea_+10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': 0.1},
    'Urea_-10%': {'Annual_Total_Precip': 0, 'District_Urea_MT': -0.1}
}
scenario_results = {}

for scenario, changes in scenarios.items():
    X_temp = X_scenario.copy()
    for feature, change in changes.items():
        X_temp[feature] = X_scenario[feature] * (1 + change)
    scenario_results[scenario] = paddy_model.predict(X_temp).mean()

plt.figure(figsize=(8, 6))
plt.bar(scenario_results.keys(), scenario_results.values())
plt.ylabel('Predicted Paddy Yield (t/ha)')
plt.title('Scenario Analysis: Rainfall and Urea Impact')
plt.xticks(rotation=45)
plt.savefig(base_path + 'scenario_analysis.png')
plt.close()

# Step 10: District-Specific Yield Predictions for 2023-2024
for crop in target_crops:
    model = results[crop]['Best_Model']
    data_2023_2024[f'{crop}_Predicted'] = model.predict(data_2023_2024[features])
    data_2023_2024[f'{crop}_Error'] = data_2023_2024[f'{crop}_Predicted'] - data_2023_2024[f'{crop}']

# Save predictions
prediction_columns = ['District', 'Year', 
                     'Paddy_Yield', 'Paddy_Yield_Predicted', 'Paddy_Yield_Error',
                     'Maize_Yield', 'Maize_Yield_Predicted', 'Maize_Yield_Error',
                     'Wheat_Yield', 'Wheat_Yield_Predicted', 'Wheat_Yield_Error']
data_2023_2024[prediction_columns].to_csv(base_path + 'district_predictions_2023_2024.csv', index=False)

# Visualize top 10 districts by predicted yield
for crop in target_crops:
    top_districts = data_2023_2024[['District', f'{crop}_Predicted']].sort_values(by=f'{crop}_Predicted', ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=f'{crop}_Predicted', y='District', data=top_districts)
    plt.xlabel(f'Predicted {crop} (t/ha)')
    plt.title(f'Top 10 Districts by Predicted {crop} (2023-2024)')
    plt.savefig(base_path + f'district_yield_predictions_{crop}.png')
    plt.close()

# Visualize top 10 districts by prediction error
for crop in target_crops:
    top_errors = data_2023_2024[['District', f'{crop}_Error']].sort_values(by=f'{crop}_Error', key=abs, ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=f'{crop}_Error', y='District', data=top_errors)
    plt.xlabel(f'Prediction Error for {crop} (t/ha)')
    plt.title(f'Top 10 Districts by {crop} Prediction Error (2023-2024)')
    plt.savefig(base_path + f'district_yield_error_{crop}.png')
    plt.close()

# Step 11: Save Chart.js for Presentation
chartjs_config = {
    "type": "bar",
    "data": {
        "labels": ["Paddy", "Maize", "Wheat"],
        "datasets": [
            {"label": "RMSE (t/ha)", "data": rmse_values, "backgroundColor": "#1f77b4"},
            {"label": "R²", "data": r2_values, "backgroundColor": "#ff7f0e"}
        ]
    },
    "options": {
        "scales": {
            "y": {"beginAtZero": True, "title": {"display": True, "text": "Value"}},
            "x": {"title": {"display": True, "text": "Crop"}}
        },
        "plugins": {"title": {"display": True, "text": "Model Performance: RMSE and R²"}}
    }
}
with open(base_path + 'model_performance_chart.json', 'w') as f:
    json.dump(chartjs_config, f, indent=2)

# Save processed data
data.to_csv(base_path + 'processed_data.csv', index=False)
 



Unique districts in 2023-2024 agricultural data: 77
Districts: ['ACHHAM', 'ARGHAKHANCHI', 'BAGLUNG', 'BAITADI', 'BAJHANG', 'BAJURA', 'BANKE', 'BARA', 'BARDIYA', 'BHAKTAPUR', 'BHOJPUR', 'CHITWAN', 'DADELDHURA', 'DAILEKH', 'DANG', 'DARCHULA', 'DHADING', 'DHANKUTA', 'DHANUSHA', 'DOLAKHA', 'DOLPA', 'DOTI', 'GORKHA', 'GULMI', 'HUMLA', 'ILLAM', 'JAJARKOT', 'JHAPA', 'JUMLA', 'KAILALI', 'KALIKOT', 'KANCHANPUR', 'KAPILBASTU', 'KASKI', 'KATHMANDU', 'KAVRE', 'KHOTANG', 'LALITPUR', 'LAMJUNG', 'MAHOTTARI', 'MAKWANPUR', 'MANANG', 'MORANG', 'MUGU', 'MUSTANG', 'MYAGDI', 'NAWALPARASI EAST', 'NAWALPARASI WEST', 'NUWAKOT', 'OKHALDHUNGA', 'PALPA', 'PANCHTHAR', 'PARBAT', 'PARSA', 'PYUTHAN', 'RAMECHAP', 'RASUWA', 'RAUTAHAT', 'ROLPA', 'RUKUM EAST', 'RUKUM WEST', 'RUPANDEHI', 'SALYAN', 'SANKHUWASABHA', 'SAPTARI', 'SARLAHI', 'SINDHULI', 'SINDHUPALCHOK', 'SIRAHA', 'SOLUKHUMBU', 'SUNSARI', 'SURKHET', 'SYANGJA', 'TANAHU', 'TAPLEJUNG', 'TERHATHUM', 'UDAYAPUR']
After deduplication, districts in 2023-2024: 77
Unique

In [11]:

# Step 8: Model Evaluation and Feature Importance
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'results', 'features', and 'base_path' are defined from your code
feature_importance_dict = {}
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    model = results[crop]['Best_Model']
    y_pred = model.predict(results[crop]['X_test'])
    rmse = np.sqrt(mean_squared_error(results[crop]['y_test'], y_pred))
    r2 = r2_score(results[crop]['y_test'], y_pred)
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    
    print(f"\n{crop}:")
    print(f"  RMSE: {rmse:.3f} t/ha")
    print(f"  R²: {r2:.3f}")
    print(f"  Cross-Validation R²: {cv_scores.mean():.3f} (±{cv_scores.std():.3f})")
    print(f"  Best Parameters: {results[crop]['Best_Params']}")
    
    # Feature Importance
    importance = pd.DataFrame({
        'Feature': features,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    feature_importance_dict[crop] = importance
    print(f"  Feature Importance for {crop}:\n{importance.head(5)}")  # Top 5 features
    
    # Save to CSV
    importance.to_csv(base_path + f'feature_importance_{crop}.csv', index=False)
    
    # Visualize Feature Importance
    plt.figure(figsize=(8, 6))
    sns.barplot(x='Importance', y='Feature', data=importance.head(6), palette='viridis')
    plt.title(f'Top Features for {crop} Prediction')
    plt.xlabel('Importance Score')
    plt.savefig(base_path + f'feature_importance_{crop}_top6.png', bbox_inches='tight')
    plt.close()

# Save combined feature importance for all crops
combined_importance = pd.concat(
    [feature_importance_dict[crop].assign(Crop=crop.split('_')[0]) for crop in target_crops],
    ignore_index=True
)
combined_importance.to_csv(base_path + 'combined_feature_importance.csv', index=False)



Paddy_Yield:
  RMSE: 0.257 t/ha
  R²: 0.899
  Cross-Validation R²: 0.770 (±0.143)
  Best Parameters: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
  Feature Importance for Paddy_Yield:
             Feature  Importance
10         Longitude    0.305955
9           Latitude    0.299878
0   Annual_Mean_Temp    0.206886
6      Paddy_Area_ha    0.044285
7      Maize_Area_ha    0.032266

Maize_Yield:
  RMSE: 0.388 t/ha
  R²: 0.753
  Cross-Validation R²: 0.770 (±0.143)
  Best Parameters: {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
  Feature Importance for Maize_Yield:
               Feature  Importance
9             Latitude    0.326057
10           Longitude    0.205149
0     Annual_Mean_Temp    0.127757
5   District_Potash_MT    0.096555
6        Paddy_Area_ha    0.048152

Wheat_Yield:
  RMSE: 0.263 t/ha
  R²: 0.828
  Cross-Validation R²: 0.769 (±0.142)
  Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
  Feature Importanc

In [13]:

# Enhanced Step 8: Combined Feature Importance Plot
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'results', 'features', 'base_path' are defined
feature_importance_dict = {}
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    model = results[crop]['Best_Model']
    importance = pd.DataFrame({
        'Feature': features,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    feature_importance_dict[crop] = importance
    print(f"\nTop 5 Features for {crop}:\n{importance.head(5)[['Feature', 'Importance']]}")
    importance.to_csv(base_path + f'feature_importance_{crop}.csv', index=False)

# Combined plot for top 5 features
top_features = set()
for crop in feature_importance_dict:
    top_features.update(feature_importance_dict[crop]['Feature'].head(5))
top_features = list(top_features)

plt.figure(figsize=(10, 6))
for crop in ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']:
    importance = feature_importance_dict[crop]
    top_data = importance[importance['Feature'].isin(top_features)][['Feature', 'Importance']]
    sns.barplot(x='Feature', y='Importance', hue=crop.split('_')[0], data=top_data)
plt.title('Top Factors Affecting Crop Yields')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Crop')
plt.tight_layout()
plt.savefig(base_path + 'combined_feature_importance.png', bbox_inches='tight')
plt.close()



Top 5 Features for Paddy_Yield:
             Feature  Importance
10         Longitude    0.305955
9           Latitude    0.299878
0   Annual_Mean_Temp    0.206886
6      Paddy_Area_ha    0.044285
7      Maize_Area_ha    0.032266

Top 5 Features for Maize_Yield:
               Feature  Importance
9             Latitude    0.326057
10           Longitude    0.205149
0     Annual_Mean_Temp    0.127757
5   District_Potash_MT    0.096555
6        Paddy_Area_ha    0.048152

Top 5 Features for Wheat_Yield:
             Feature  Importance
0   Annual_Mean_Temp    0.495076
9           Latitude    0.165652
10         Longitude    0.117271
7      Maize_Area_ha    0.070266
8      Wheat_Area_ha    0.039230


ValueError: Could not interpret value `Paddy` for `hue`. An entry with this name does not appear in `data`.

<Figure size 1000x600 with 0 Axes>

In [14]:

# Test Paddy Model Without Latitude/Longitude
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data', 'features', 'results', and 'base_path' are defined from your code
features_no_geo = [f for f in features if f not in ['Latitude', 'Longitude']]
X = data[features_no_geo]
y = data['Paddy_Yield']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(**results['Paddy_Yield']['Best_Params'], random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
r2_no_geo = r2_score(y_test, y_pred)
rmse_no_geo = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"\nPaddy Model Without Geographic Features:")
print(f"R²: {r2_no_geo:.3f}")
print(f"RMSE: {rmse_no_geo:.3f} t/ha")

# Feature Importance
importance = pd.DataFrame({
    'Feature': features_no_geo,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(f"Top 5 Features:\n{importance.head(5)}")
importance.to_csv(base_path + 'feature_importance_paddy_no_geo.csv', index=False)

# Visualize
plt.figure(figsize=(8, 5))
sns.barplot(x='Importance', y='Feature', data=importance.head(5), palette='viridis')
plt.title('Top 5 Features for Paddy (No Geographic Features)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig(base_path + 'feature_importance_paddy_no_geo.png', bbox_inches='tight')
plt.close()



Paddy Model Without Geographic Features:
R²: 0.514
RMSE: 0.564 t/ha
Top 5 Features:
                Feature  Importance
0      Annual_Mean_Temp    0.432127
1   Annual_Total_Precip    0.118268
2  Monsoon_Total_Precip    0.115846
6         Paddy_Area_ha    0.079459
7         Maize_Area_ha    0.075529


In [15]:

# Test Models Without Latitude/Longitude for Paddy, Maize, and Wheat
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data', 'features', 'results', and 'base_path' are defined from your code
features_no_geo = [f for f in features if f not in ['Latitude', 'Longitude']]
target_crops = ['Paddy_Yield', 'Maize_Yield', 'Wheat_Yield']

for crop in target_crops:
    print(f"\nTesting Model for {crop} Without Geographic Features:")
    # Prepare data
    X = data[features_no_geo]
    y = data[crop]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train model with best parameters
    rf = RandomForestRegressor(**results[crop]['Best_Params'], random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    # Calculate metrics
    r2_no_geo = r2_score(y_test, y_pred)
    rmse_no_geo = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"R²: {r2_no_geo:.3f}")
    print(f"RMSE: {rmse_no_geo:.3f} t/ha")

    # Feature Importance
    importance = pd.DataFrame({
        'Feature': features_no_geo,
        'Importance': rf.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    print(f"Top 5 Features:\n{importance.head(5)}")
    
    # Save to CSV
    importance.to_csv(base_path + f'feature_importance_{crop}_no_geo.csv', index=False)

    # Visualize
    plt.figure(figsize=(8, 5))
    sns.barplot(x='Importance', y='Feature', data=importance.head(5), palette='viridis')
    plt.title(f'Top 5 Features for {crop} (No Geographic Features)')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig(base_path + f'feature_importance_{crop}_no_geo.png', bbox_inches='tight')
    plt.close()


Testing Model for Paddy_Yield Without Geographic Features:
R²: 0.514
RMSE: 0.564 t/ha
Top 5 Features:
                Feature  Importance
0      Annual_Mean_Temp    0.432127
1   Annual_Total_Precip    0.118268
2  Monsoon_Total_Precip    0.115846
6         Paddy_Area_ha    0.079459
7         Maize_Area_ha    0.075529

Testing Model for Maize_Yield Without Geographic Features:
R²: 0.505
RMSE: 0.550 t/ha
Top 5 Features:
                Feature  Importance
0      Annual_Mean_Temp    0.350416
2  Monsoon_Total_Precip    0.144149
5    District_Potash_MT    0.106721
6         Paddy_Area_ha    0.093728
1   Annual_Total_Precip    0.079558

Testing Model for Wheat_Yield Without Geographic Features:
R²: 0.606
RMSE: 0.398 t/ha
Top 5 Features:
                Feature  Importance
0      Annual_Mean_Temp    0.583759
1   Annual_Total_Precip    0.090255
7         Maize_Area_ha    0.087129
2  Monsoon_Total_Precip    0.085145
8         Wheat_Area_ha    0.060618
